In [ ]:
from scipy.optimize import linear_sum_assignment
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
import os
import shutil
import yaml
from ultralytics import YOLO
import cv2
import matplotlib.patches as patches
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from glob import glob
import torch
from torchvision import transforms
from PIL import Image
import sys
# Add the csrnet directory to the path to import the model
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), 'csrnet')))
from csrnet_model import CSRNet

In [ ]:
# Load results from csv
results_path = '<DATASET_ROOT>/Exp_06_Vorversuche_YOLO_CSR/runs/detect_11x/train/results.csv'
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    # The column names can have leading/trailing spaces, let's strip them
    results_df.columns = results_df.columns.str.strip()
    print("YOLO training results loaded successfully.")
    display(results_df.head())
else:
    print(f"Could not find results file at: {results_path}")

model_path = '<DATASET_ROOT>/Exp_06_Vorversuche_YOLO_CSR/runs/detect_11x/train/weights/best.pt'
csrnet_model_path = '<DATASET_ROOT>/Exp_06_Vorversuche_YOLO_CSR/runs/csrnet_train/best_model.pth'


In [ ]:
# --- Configuration ---
# This should be the main folder for your experiment
base_dir = '<DATASET_ROOT>/Exp_06_Vorversuche_YOLO_CSR/'
db_path = os.path.join(base_dir, 'stripGen_results.db')
output_dataset_dir = os.path.join(base_dir, 'dataset')

# --- 1. Load Data ---
print(f"Loading data from {db_path}...")
sqlite_connection = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM results", sqlite_connection)
sqlite_connection.close()

# Make sure image paths are absolute
df['image_path'] = df['image_path'].apply(lambda p: os.path.join(base_dir, p))
all_image_paths = df['image_path'].tolist()

In [ ]:
# Find index for the specified parameters
query = (
    (df['random_seed'] == 70) &
    (df['belt_speed'] == -0.01) &
    (df['measurement_frame'] == 1050) &
    (df['number_of_strips'] == 1000)
)
matching_indices = df.index[query].tolist()
print("Matching indices:", matching_indices)

## Compare Trained Model with Ground Truth

In [ ]:
def calculate_iou(box1, box2):
    """
    Calculates the Intersection over Union (IoU) of two bounding boxes.
    
    Args:
        box1, box2: [x1, y1, x2, y2]
    
    Returns:
        iou (float): The IoU value.
    """
    # Determine the coordinates of the intersection rectangle
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    # The area of intersection
    intersection_area = (x_right - x_left) * (y_bottom - y_top)

    # The area of both bounding boxes
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # The area of union
    union_area = box1_area + box2_area - intersection_area

    # Compute the IoU
    iou = intersection_area / union_area
    return iou

def plot_comparison(idx, iou_threshold=0.5):
    """
    Plots a side-by-side comparison of ground truth and predicted bounding boxes.
    """
    image_path = Path(df['image_path'].iloc[idx])
    gt_bbox_path = image_path.with_suffix('.txt')
    print(f"Processing image: {image_path.name}")
    print(f"belt_speed: {df['belt_speed'].iloc[idx]}")
    print(f"number_of_strips: {df['number_of_strips'].iloc[idx]}")
    # --- Load Image ---
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img_rgb.shape

    # --- Get Ground Truth Boxes ---
    gt_boxes = []
    if gt_bbox_path.exists():
        with open(gt_bbox_path, 'r') as f:
            for line in f.readlines():
                _, cx, cy, w, h = map(float, line.strip().split())
                x1 = (cx - w / 2) * width
                y1 = (cy - h / 2) * height
                x2 = (cx + w / 2) * width
                y2 = (cy + h / 2) * height
                gt_boxes.append([x1, y1, x2, y2])

    # --- Get Predicted Boxes ---
    if not os.path.exists(model_path):
        print(f"Model not found at {model_path}. Please ensure the model is trained and the path is correct.")
        return
        
    model = YOLO(model_path)
    results = model(image_path, verbose=False)
    pred_boxes = results[0].boxes.xyxy.cpu().numpy().tolist()

    # --- Match Boxes using IoU and Hungarian Algorithm ---
    iou_matrix = np.zeros((len(gt_boxes), len(pred_boxes)))
    for i, gt_box in enumerate(gt_boxes):
        for j, pred_box in enumerate(pred_boxes):
            iou_matrix[i, j] = calculate_iou(gt_box, pred_box)

    # Use Hungarian algorithm for optimal assignment
    gt_indices, pred_indices = linear_sum_assignment(-iou_matrix)
    
    matched_gt = set()
    matched_pred = set()
    
    for gt_idx, pred_idx in zip(gt_indices, pred_indices):
        if iou_matrix[gt_idx, pred_idx] >= iou_threshold:
            matched_gt.add(gt_idx)
            matched_pred.add(pred_idx)

    # --- Visualization ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    # Plot Ground Truth
    ax1.imshow(img_rgb)
    ax1.set_title(f'Ground Truth ({len(gt_boxes)} boxes)')
    ax1.axis('off')
    for i, box in enumerate(gt_boxes):
        color = 'lime' if i in matched_gt else 'red'
        rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=2, edgecolor=color, facecolor='none')
        ax1.add_patch(rect)

    # Plot Predictions
    ax2.imshow(img_rgb)
    ax2.set_title(f'YOLO Predictions ({len(pred_boxes)} boxes)')
    ax2.axis('off')
    for i, box in enumerate(pred_boxes):
        color = 'cyan' if i in matched_pred else 'magenta'
        rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=2, edgecolor=color, facecolor='none')
        ax2.add_patch(rect)
        
    plt.tight_layout()
    plt.show()





In [ ]:
# --- Create Interactive Widget ---
slider = widgets.IntSlider(value=10, min=0, max=len(df)-1, step=1, description='Image Index:')
interactive_plot = widgets.interactive(plot_comparison, idx=slider, iou_threshold=(0.1, 1.0, 0.05))
display(interactive_plot)

## Inference on a folder of Images

## Interactively Crop Images for Testing
Use the widgets below to select an image and define crop boundaries. The original and cropped images will be displayed side-by-side. This helps in finding the right coordinates for batch processing.

Good intial values are:
X: 225,550
Y: 100,425

In [ ]:
# specify the results directory
image_folder='<DATASET_ROOT>/05_Experimente/0meterpm35upm99fps10msshutter_20sec/DOLP'
model_image_res=(512, 512)

In [ ]:
# --- Interactive Widget for Cropping ---

def plot_cropped_image(image_path, x_min=250, y_min=100, x_max=550, y_max=425):
    """
    Loads an image, crops it, and displays both the original and the cropped version.
    """
    if not image_path or not os.path.exists(image_path):
        print("Please select a valid image.")
        return

    # --- Load Image ---
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img_rgb.shape

    # --- Clamp crop values to image dimensions ---
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(width, x_max)
    y_max = min(height, y_max)

    if x_min >= x_max or y_min >= y_max:
        print("Invalid crop boundaries. (x_min < x_max and y_min < y_max)")
        # --- Visualization for invalid crop ---
        fig, ax1 = plt.subplots(1, 1, figsize=(10, 10))
        ax1.imshow(img_rgb)
        ax1.set_title('Original Image - Invalid Crop')
        ax1.axvline(x_min, color='r', linestyle='--')
        ax1.axvline(x_max, color='r', linestyle='--')
        ax1.axhline(y_min, color='r', linestyle='--')
        ax1.axhline(y_max, color='r', linestyle='--')
        ax1.axis('off')
        plt.show()
        return

    # --- Crop Image ---
    cropped_img = img_rgb[y_min:y_max, x_min:x_max]

    # --- Visualization ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    # Original Image
    ax1.imshow(img_rgb)
    ax1.set_title('Original Image')
    ax1.axvline(x_min, color='r', linestyle='--')
    ax1.axvline(x_max, color='r', linestyle='--')
    ax1.axhline(y_min, color='r', linestyle='--')
    ax1.axhline(y_max, color='r', linestyle='--')
    ax1.axis('off')

    # Cropped Image
    ax2.imshow(cropped_img)
    ax2.set_title('Cropped Image')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

# --- Create Widgets ---
# Use a different name to avoid conflicts with the inference widget
folder_path_widget_crop = widgets.Text(
    value=image_folder,
    placeholder='Enter path to image folder',
    description='Image Folder:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

image_selector_widget_crop = widgets.Dropdown(
    description='Select Image:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

x_min_widget = widgets.IntText(value=0, description='X Min:', style={'description_width': 'initial'})
y_min_widget = widgets.IntText(value=0, description='Y Min:', style={'description_width': 'initial'})
x_max_widget = widgets.IntText(value=512, description='X Max:', style={'description_width': 'initial'})
y_max_widget = widgets.IntText(value=512, description='Y Max:', style={'description_width': 'initial'})

crop_box = widgets.HBox([x_min_widget, y_min_widget, x_max_widget, y_max_widget])

# An output widget to display the plot
output_widget_crop = widgets.Output()

# --- Link Widgets ---
def update_image_list_crop(change):
    # Use key access for dict, attribute access for traitlets
    folder_path = change['new'] if isinstance(change, dict) else change.new
    if os.path.isdir(folder_path):
        image_files = sorted(glob(os.path.join(folder_path, '*.png')) + \
                             glob(os.path.join(folder_path, '*.bmp')) + \
                             glob(os.path.join(folder_path, '*.jpg')) + \
                             glob(os.path.join(folder_path, '*.jpeg')))
        image_selector_widget_crop.options = [(os.path.basename(f), f) for f in image_files]
    else:
        image_selector_widget_crop.options = []

def on_crop_params_change(change):
    with output_widget_crop:
        clear_output(wait=True)
        plot_cropped_image(
            image_selector_widget_crop.value,
            x_min_widget.value,
            y_min_widget.value,
            x_max_widget.value,
            y_max_widget.value
        )

# --- Observe changes ---
folder_path_widget_crop.observe(update_image_list_crop, names='value')
image_selector_widget_crop.observe(on_crop_params_change, names='value')
x_min_widget.observe(on_crop_params_change, names='value')
y_min_widget.observe(on_crop_params_change, names='value')
x_max_widget.observe(on_crop_params_change, names='value')
y_max_widget.observe(on_crop_params_change, names='value')


# --- Display Widgets & Initial Population ---
display(widgets.VBox([folder_path_widget_crop, image_selector_widget_crop, crop_box, output_widget_crop]))

# Manually trigger the update to populate the list initially
# Check if the folder path widget has a value before triggering
if folder_path_widget_crop.value:
    update_image_list_crop({'new': folder_path_widget_crop.value})


### Inference on Real-Images

In [ ]:
# --- Interactive Widget for Inference on a Folder ---

# --- Configuration for Inference Widget ---
yolo_model = None
if not os.path.exists(model_path):
    print(f"YOLO model not found at {model_path}. Please train the model first.")
else:
    yolo_model = YOLO(model_path)
    print(f"Loaded YOLO model from {model_path}")

csrnet_model = None
if not os.path.exists(csrnet_model_path):
    print(f"CSRNet model not found at {csrnet_model_path}. Please train the model first.")
else:
    # --- Load CSRNet Model ---
    # The model was trained on GPU, so we need to map it to CPU if CUDA is not available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    csrnet_model = CSRNet()
    # Load the state dict before wrapping with DataParallel
    csrnet_model.load_state_dict(torch.load(csrnet_model_path, map_location=device))
    # The model is wrapped in DataParallel to match training setup
    csrnet_model = torch.nn.DataParallel(csrnet_model)
    csrnet_model = csrnet_model.to(device)
    csrnet_model.eval() # Set to evaluation mode
    print(f"Loaded CSRNet model from {csrnet_model_path}")

def plot_predictions(image_path, x_min, y_min, x_max, y_max, model_type):
    """
    Crops an image, runs the selected model on the crop, and plots predictions.
    """
    if (model_type == 'YOLO' and not yolo_model) or \
       (model_type == 'CSRNet' and not csrnet_model):
        print(f"{model_type} model is not loaded.")
        return
        
    if not image_path or not os.path.exists(image_path):
        print("Image not found or not selected.")
        return

    # --- Load Image ---
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img_rgb.shape

    # --- Clamp crop values to image dimensions ---
    x_min_c = max(0, x_min)
    y_min_c = max(0, y_min)
    x_max_c = min(width, x_max)
    y_max_c = min(height, y_max)

    if x_min_c >= x_max_c or y_min_c >= y_max_c:
        print("Invalid crop boundaries.")
        return

    # --- Crop Image for inference ---
    cropped_img_rgb = img_rgb[y_min_c:y_max_c, x_min_c:x_max_c]
    
    # --- Visualization ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    # Original Image with crop box
    ax1.imshow(img_rgb)
    ax1.set_title('Original Image')
    ax1.axis('off')
    crop_rect = patches.Rectangle((x_min_c, y_min_c), x_max_c - x_min_c, y_max_c - y_min_c, linewidth=1, edgecolor='r', linestyle='--', facecolor='none')
    ax1.add_patch(crop_rect)

    # --- Model-specific Inference and Plotting ---
    if model_type == 'YOLO':
        results = yolo_model(cropped_img_rgb, verbose=False)
        pred_boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
        
        ax2.imshow(cropped_img_rgb)
        ax2.set_title(f'YOLO Predictions ({len(pred_boxes)} boxes) on cropped area')
        ax2.axis('off')
        for box in pred_boxes:
            rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=2, edgecolor='cyan', facecolor='none')
            ax2.add_patch(rect)

    elif model_type == 'CSRNet':
        # --- Preprocess for CSRNet ---
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        pil_img = Image.fromarray(cropped_img_rgb)
        img_tensor = transform(pil_img).unsqueeze(0)
        
        # --- Get Density Map ---
        with torch.no_grad():
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            img_tensor = img_tensor.to(device)
            density_map = csrnet_model(img_tensor)
            count = int(density_map.sum().item())
        
        # --- Visualization ---
        density_map_np = density_map.squeeze(0).squeeze(0).cpu().numpy()
        ax2.imshow(density_map_np, cmap='jet')
        ax2.set_title(f'CSRNet Density Map (Count: {count}) on cropped area')
        ax2.axis('off')
        
    plt.tight_layout()
    plt.show()

# --- Create Widgets ---
model_selector_widget = widgets.Dropdown(
    options=['YOLO', 'CSRNet'],
    value='YOLO',
    description='Model:',
    style={'description_width': 'initial'}
)

folder_path_widget = widgets.Text(
    value=image_folder,
    placeholder='Enter path to image folder',
    description='Image Folder:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

image_selector_widget = widgets.Dropdown(
    description='Select Image:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%')
)

x_min_inf_widget = widgets.IntText(value=x_min_widget.value, description='X Min:', style={'description_width': 'initial'})
y_min_inf_widget = widgets.IntText(value=y_min_widget.value, description='Y Min:', style={'description_width': 'initial'})
x_max_inf_widget = widgets.IntText(value=x_max_widget.value, description='X Max:', style={'description_width': 'initial'})
y_max_inf_widget = widgets.IntText(value=y_max_widget.value, description='Y Max:', style={'description_width': 'initial'})

crop_box_inf = widgets.HBox([x_min_inf_widget, y_min_inf_widget, x_max_inf_widget, y_max_inf_widget])

output_widget = widgets.Output()

# --- Link Widgets ---
def update_image_list(change):
    folder_path = change['new']
    if os.path.isdir(folder_path):
        image_files = sorted(glob(os.path.join(folder_path, '*.png')) + \
                             glob(os.path.join(folder_path, '*.bmp')) + \
                             glob(os.path.join(folder_path, '*.jpg')) + \
                             glob(os.path.join(folder_path, '*.jpeg')))
        image_selector_widget.options = [(os.path.basename(f), f) for f in image_files]
    else:
        image_selector_widget.options = []

def on_params_change(change):
    with output_widget:
        clear_output(wait=True)
        if image_selector_widget.value:
            plot_predictions(
                image_selector_widget.value,
                x_min_inf_widget.value,
                y_min_inf_widget.value,
                x_max_inf_widget.value,
                y_max_inf_widget.value,
                model_selector_widget.value
            )

# --- Observe changes ---
model_selector_widget.observe(on_params_change, names='value')
folder_path_widget.observe(update_image_list, names='value')
image_selector_widget.observe(on_params_change, names='value')
x_min_inf_widget.observe(on_params_change, names='value')
y_min_inf_widget.observe(on_params_change, names='value')
x_max_inf_widget.observe(on_params_change, names='value')
y_max_inf_widget.observe(on_params_change, names='value')

# --- Display Widgets & Initial Population ---
display(widgets.VBox([model_selector_widget, folder_path_widget, image_selector_widget, crop_box_inf, output_widget]))

# Manually trigger the update to populate the list initially
if folder_path_widget.value:
    update_image_list({'new': folder_path_widget.value})
